# W04 -- Exploratory Data Analysis on Housing Data

Python EDA companion to `W04-exploration-housing.rmd` (the R version of this same analysis). Covers:

1. Zillow Home Value Index (ZHVI): reshaping to long format, national trend, missingness, seasonality, distribution, and by-region spread.
2. FHFA Home Price Index (HPI) at the metro level: cleaning, isolating the three training cities (Austin, Boise, Tampa), and a missingness check.

**Inputs:** `data/zhvi_raw.csv`, `data/hpi_at_metro.csv`, `data/cbsa-est2025-alldata.csv`, `data/cbsa-est2019-alldata.csv`.

**Outputs:** PNGs under `data/zillow_plots/` and `data/hpi_at_metro_plots/`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import seaborn as sns

In [ ]:
zillow_data = pd.read_csv("../../data/zhvi_raw.csv")
hpi_at_metro = pd.read_csv("../../data/hpi_at_metro.csv")

# Population estimates -- loaded here to confirm availability; used later in
# feature engineering (population velocity/acceleration), not in this EDA notebook.
cbsa_2025 = pd.read_csv("../../data/cbsa-est2025-alldata.csv", encoding='iso-8859-1')
cbsa_2019 = pd.read_csv("../../data/cbsa-est2019-alldata.csv", encoding='iso-8859-1')

## Reshape Zillow data to long format

The raw file has one column per month (wide format), which makes it awkward to plot or join. We melt it down to one row per region/date.

In [ ]:
dates_cols = [c for c in zillow_data.columns if not pd.isna(pd.to_datetime(c, errors='coerce'))]
id_cols = [c for c in zillow_data.columns if c not in dates_cols]

zillow_long = zillow_data.melt(id_vars=id_cols, value_vars=dates_cols, var_name='Date', value_name='Value')
zillow_long['Date'] = pd.to_datetime(zillow_long['Date'])

print(zillow_long.head())
print(zillow_long.columns.tolist())

## Zillow EDA

National time series, missingness, seasonality, price distribution, and spread across the top 15 regions by average price.

In [ ]:
ts_national = zillow_long.groupby('Date')['Value'].mean().reset_index()
plt.figure(figsize=(12, 5))
plt.plot(ts_national['Date'], ts_national['Value'])
plt.title('National Average Zillow Prices Over Time')
plt.xlabel('Date')
plt.ylabel('Price')
plt.tight_layout()
plt.savefig('../../data/zillow_plots/timeseriesplot.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.heatmap(zillow_long.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.xlabel('Date')
plt.ylabel('Region')
plt.tight_layout()
plt.savefig('../../data/zillow_plots/missing_values_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
zillow_long['month'] = zillow_long['Date'].dt.month
seasonality = zillow_long.groupby('month')['Value'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=seasonality, x='month', y='Value')
plt.title('Average Zillow Prices by Month')
plt.xlabel('Month')
plt.ylabel('Average Price')
plt.xticks(ticks=range(1, 13), labels=[
    'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'
])
plt.tight_layout()
plt.savefig('../../data/zillow_plots/seasonality_plot.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(zillow_long['Value'], bins=50, kde=True)
plt.title('Distribution of Zillow Prices')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('../../data/zillow_plots/distribution_plot.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
top_regions = zillow_long.groupby('RegionName')['Value'].mean().nlargest(15).index
long_top = zillow_long[zillow_long['RegionName'].isin(top_regions)]

plt.figure(figsize=(14, 7))
sns.boxplot(data=long_top, x='RegionName', y='Value', hue='RegionName', palette='Set2', legend=False)
plt.title('Zillow Prices by Region (Top 15)', fontsize=14, fontweight='bold')
plt.xlabel('Region', fontsize=12)
plt.ylabel('Price ($)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../../data/zillow_plots/boxplot_by_state.png', dpi=100, bbox_inches='tight')
plt.show()

## House Price Index at metro level

The raw HPI file has no column headers, so we add them, then clean up the `Value` and `Difference` columns (which arrive as strings using `$`, commas, parentheses for negatives, and `-` for missing/zero).

In [ ]:
# Adds headers and writes back to the source file so downstream notebooks/scripts
# can read data/hpi_at_metro.csv directly with headers already in place.
hpi_at_metro.columns = ['Metro', 'Area Code', 'Year', 'Quarter', 'Value', 'Difference']
hpi_at_metro.to_csv('../../data/hpi_at_metro.csv', index=False)
hpi_at_metro = pd.read_csv('../../data/hpi_at_metro.csv')

training_cities = hpi_at_metro[hpi_at_metro['Metro'].isin(
    ['Boise City, ID', 'Tampa, FL (MSAD)', 'Austin-Round Rock-San Marcos, TX']
)]
print(training_cities.head())
print(training_cities.dtypes)

In [ ]:
# Value and Difference arrive as strings. A naive $/,-strip fails because missing/zero
# values use a dash ("-") instead of NaN/0, so we replace the dash with "0" first,
# and for Difference we also need to strip the parentheses used for negative values.
hpi_at_metro['Value'] = (
    hpi_at_metro['Value'].str.replace('-', '0').str.replace('$', '').str.replace(',', '').astype(float)
)
hpi_at_metro['Difference'] = (
    hpi_at_metro['Difference'].str.replace('-', '0').str.replace('(', '').str.replace(')', '')
    .str.replace('$', '').str.replace(',', '').str.strip().astype(float)
)

In [ ]:
plt.figure(figsize=(14, 6))
sns.heatmap(hpi_at_metro.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.xlabel('Date')
plt.ylabel('Region')
plt.tight_layout()
plt.savefig('../../data/hpi_at_metro_plots/missing_values_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

for city in ['Boise City, ID', 'Tampa, FL (MSAD)', 'Austin-Round Rock-San Marcos, TX']:
    city_data = hpi_at_metro[hpi_at_metro['Metro'] == city]
    city_data = city_data.sort_values(['Year', 'Quarter'])
    plt.plot(city_data['Year'], city_data['Value'], marker='o', label=city, linewidth=2)

plt.title('HPI over Time by Training City')
plt.xlabel('Year')
plt.ylabel('HPI Value')
plt.legend()
plt.grid(True, alpha=0.3)
ax = plt.gca()
ax.yaxis.set_major_locator(MaxNLocator(nbins=10))
plt.tight_layout()
plt.savefig('../../data/hpi_at_metro_plots/hpi_over_time_by_city.png', dpi=100, bbox_inches='tight')
plt.show()